# MLB game simulator: 2026 retrospective backtest

Simulates each MLB regular-season game one plate appearance at a time, about 1,000 times, and turns the share of simulated wins into each team's win probability. Follows `project_deliverables/MLB_Game_Simulator_Implementation_Plan.docx`.

**Workflow.** Build and tune on 2022-2025: settings are chosen on 2024 using only 2022-2023, then checked once on 2025 using 2022-2024. With the settings frozen, every table is rebuilt from 2022-2025 and each completed 2026 game is simulated from its actual starting lineups, starting pitchers and pregame bullpen list.

**What stays out of predictions.** The 2026 season is over, but its scores, winners and full-season player totals never enter a prediction. The 2026 predictions are saved and hashed before `game_results.csv` is read for 2026. Because lineups and starters come from box scores, the test is **conditional on known starting lineups and starters**. That is easier than forecasting before lineups are announced.

**Outputs** go to `data/simulation/`. A rerun must reproduce existing files exactly, or it stops (`OVERWRITE_RESULTS`). Runtime is about 15 minutes.

## Imports and configuration

In [1]:
import hashlib
import io
import json
import platform
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import optimize, special

# Find the repository root so paths are the same whether Jupyter starts there or in notebooks/.
REPO_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                  if (folder / "requirements.txt").is_file()
                  and (folder / "notebooks" / "game_simulator.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"Could not find the repository root from {Path.cwd().name}/: no folder above it contains "
                            "requirements.txt and notebooks/game_simulator.ipynb. Start Jupyter inside the repository.")
GAME_DIR = REPO_ROOT / "data" / "games"        # collected game tables (inputs; never written here)
RAW_DIR = REPO_ROOT / "data" / "raw"           # only mlb_player_handedness.csv is used; season totals are not
OUT_DIR = REPO_ROOT / "data" / "simulation"    # predictions, evaluation tables and figures from this notebook
OVERWRITE_RESULTS = False   # False: an existing output must match the rerun exactly, or the run stops

DEV_SEASONS = [2022, 2023, 2024, 2025]
TUNE_SEASON, CHECK_SEASON, TEST_SEASON = 2024, 2025, 2026
EVENT_FIRST_SEASON = 2023   # bigger bases and pickoff limits changed steals in 2023, so earlier events aren't used
N_SIMS = 1000               # simulations per game
SEED = 23                   # each game's generator is seeded from (SEED, game_pk); both bullpen variants share draws

# Settings compared on 2024 plate appearances. A decay of 1.0 is equal weight per plate appearance; lower values
# down-weight older seasons. The latest season always has weight 1, so rescaling every weight changes nothing.
WEIGHT_DECAYS = [1.0, 0.8, 0.6]
# Multipliers on the blending strengths estimated from each training window (see "Player rates and log5").
BLEND_SCALES = [0.5, 1.0, 2.0]

# Fixed design choices, set before any evaluation and not tuned.
MIN_PLAY_CELL = 30          # runner-table cells with fewer plays borrow from the same play at any number of outs
USUAL_LENGTH_PRIOR = 3      # earlier starts' batters faced are blended with 3 league-average starts
RELIEF_SHARE_PRIOR = 20     # relief workload is blended with 20 league-average listed games
MAX_INNINGS = 60            # termination guard: a game still going after this many innings is a failure, not a tie
SMOOTHING = 0.5             # win probability = (wins + 0.5) / (N_SIMS + 1), so no game gets exactly 0 or 1
BOOTSTRAP_DRAWS = 2000      # resamples of whole game dates for intervals

OUTCOMES = ["K", "BB", "HBP", "1B", "2B", "3B", "HR", "IPNH"]   # IPNH: ball in play with no hit
K_, BB_, HBP_, S1_, S2_, S3_, HR_, IPNH_ = range(8)
EVENT_TO_OUTCOME = {
    "strikeout": "K", "strikeout_double_play": "K", "walk": "BB", "intent_walk": "BB",
    "hit_by_pitch": "HBP", "catcher_interf": "HBP", "single": "1B", "double": "2B", "triple": "3B",
    "home_run": "HR",
    **{e: "IPNH" for e in ["field_out", "force_out", "grounded_into_double_play", "double_play", "triple_play",
                           "sac_fly", "sac_fly_double_play", "sac_bunt", "sac_bunt_double_play", "fielders_choice",
                           "fielders_choice_out", "field_error"]}}
ID_COLUMNS = ["game_pk", "team_id", "home_team_id", "away_team_id", "player_id", "batter_id", "pitcher_id",
              "runner_id", "winner_team_id", "batting_team_id", "fielding_team_id", "basis_game_pk",
              *[f"{p}_on_{b}b" for p in ["start", "pre", "post", "before", "after"] for b in (1, 2, 3)]]


def read_game_csv(path, **kwargs):
    """Read a game CSV with nullable integer IDs, as the collection notebook does."""
    header = pd.read_csv(path, nrows=0).columns
    return pd.read_csv(path, dtype={c: "Int64" for c in header if c in ID_COLUMNS}, low_memory=False, **kwargs)


def save_output(content, name):
    """Write text or bytes to OUT_DIR. An existing file must be identical unless OVERWRITE_RESULTS is True."""
    data = content.encode() if isinstance(content, str) else content
    path = OUT_DIR / name
    if path.exists() and not OVERWRITE_RESULTS:
        if path.read_bytes() != data:
            raise RuntimeError(f"{path.relative_to(REPO_ROOT)} differs from this run. Inspect it, then delete it or "
                               "set OVERWRITE_RESULTS = True.")
        return "unchanged"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return "saved"


def table_text(df):
    """CSV text with fixed float formatting, so reruns produce byte-identical files."""
    return df.to_csv(index=False, float_format="%.6f", lineterminator="\n")


print(f"repository: {REPO_ROOT.name}; outputs go to {OUT_DIR.relative_to(REPO_ROOT, walk_up=True)}")
print(f"python {platform.python_version()}, numpy {np.__version__}, pandas {pd.__version__}, scipy {scipy.__version__}")

repository: mlb_game_simulator_win_probability_engine; outputs go to data\simulation
python 3.14.0, numpy 2.4.2, pandas 3.0.1, scipy 1.17.1


## Inputs and checks

- Game tables are read with nullable integer IDs, as in `savant_scrape_updated.ipynb`.
- Keys must be unique, play records must belong to 2022-2025, and every team-game needs 9 lineup slots and every game 2 starting pitchers.
- Results are kept apart from the inputs. Only 2022-2025 results load here, for development scoring and the baselines.
- Every completed plate appearance maps to exactly one of 8 outcomes: strikeout, walk (including intentional), hit by pitch (including catcher's interference), single, double, triple, home run, and **in play, no hit**. The last covers outs, double plays, sacrifices, errors and fielder's choices. A plate appearance cut off by an inning-ending runner event has no outcome.
- Games are ordered by date and doubleheader game number. `first_pitch_utc` is known only after the game, so it isn't used.

In [2]:
games = read_game_csv(GAME_DIR / "games.csv")
coverage = read_game_csv(GAME_DIR / "game_coverage.csv")
lineups = read_game_csv(GAME_DIR / "lineups.csv")
starters = read_game_csv(GAME_DIR / "starting_pitchers.csv")
bullpen = read_game_csv(GAME_DIR / "bullpen_membership.csv")
appearances = read_game_csv(GAME_DIR / "pitcher_appearances.csv")
events = read_game_csv(GAME_DIR / "between_pitch_events.csv")
pa = pd.concat([read_game_csv(GAME_DIR / f"plate_appearances_{s}.csv") for s in DEV_SEASONS], ignore_index=True)
legs = pd.concat([read_game_csv(GAME_DIR / f"runner_movements_{s}.csv") for s in DEV_SEASONS], ignore_index=True)
hands = pd.read_csv(RAW_DIR / "mlb_player_handedness.csv", usecols=["player_id", "bat_side", "pitch_hand"])
# Results are outcomes. Only development seasons are kept here; 2026 results are read in the last section,
# after the 2026 predictions are saved.
dev_results = read_game_csv(GAME_DIR / "game_results.csv").query("season in @DEV_SEASONS").reset_index(drop=True)

# Keys, seasons and lineup shape
KEYS = {"games": (games, ["game_pk"]), "coverage": (coverage, ["game_pk"]), "dev_results": (dev_results, ["game_pk"]),
        "lineups": (lineups, ["game_pk", "team_id", "batting_slot"]), "starters": (starters, ["game_pk", "team_id"]),
        "bullpen": (bullpen, ["game_pk", "team_id", "player_id"]), "appearances": (appearances, ["game_pk", "team_id", "pitcher_id"]),
        "events": (events, ["game_pk", "at_bat_index", "play_index"]), "pa": (pa, ["game_pk", "at_bat_index"]),
        "legs": (legs, ["game_pk", "at_bat_index", "play_index", "leg"]), "hands": (hands, ["player_id"])}
problems = [f"{name}: duplicate {keys}" for name, (table, keys) in KEYS.items() if table.duplicated(keys).any()]
season_of = games.set_index("game_pk")["season"]
for name in ["pa", "legs", "events", "appearances"]:
    table = KEYS[name][0]
    if not table["season"].isin(DEV_SEASONS).all() or (table["game_pk"].map(season_of) != table["season"]).any():
        problems.append(f"{name}: rows outside 2022-2025 or with a season that disagrees with games.csv")
if not (lineups.groupby(["game_pk", "team_id"]).size() == 9).all() or not (starters.groupby("game_pk").size() == 2).all():
    problems.append("every team-game needs 9 lineup slots and every game 2 starting pitchers")
if problems:
    raise ValueError("input checks failed: " + "; ".join(problems))

# Chronological order of games: date, then doubleheader game number. first_pitch_utc is post-game, so it isn't used.
games = games.sort_values(["official_date", "game_number", "game_pk"]).reset_index(drop=True)
games["game_order"] = np.arange(len(games))
order_of = games.set_index("game_pk")["game_order"]

# Outcome categories: every completed plate appearance maps to exactly one of the 8 outcomes
pa = pa.sort_values(["game_pk", "at_bat_index"]).reset_index(drop=True)
done = ~pa["is_interrupted"]
unmapped = sorted(set(pa.loc[done, "event_type"]) - set(EVENT_TO_OUTCOME))
if unmapped:
    raise ValueError(f"completed plate appearances with no outcome category: {unmapped}")
pa["outcome"] = pa["event_type"].map(EVENT_TO_OUTCOME).map({o: i for i, o in enumerate(OUTCOMES)})
pa.loc[~done, "outcome"] = np.nan          # interrupted: the batter's turn never finished
pa["game_order"] = pa["game_pk"].map(order_of)
for stage in ["start", "pre", "post"]:      # base occupancy as a 3-bit mask: 1 = first, 2 = second, 4 = third
    pa[f"{stage}_mask"] = sum(pa[f"{stage}_on_{b}b"].notna().astype(int) * (1 << (b - 1)) for b in (1, 2, 3))

# Handedness: MLB profiles first, then the side seen most often in the plays, then right-handed (counted below)
hands = hands.set_index("player_id")
seen_bat = pa.groupby("batter_id")["bat_side"].agg(lambda s: s.mode().iat[0])
seen_throw = pa.groupby("pitcher_id")["pitch_hand"].agg(lambda s: s.mode().iat[0])


def bat_side_of(ids):
    return pd.Series(ids).map(hands["bat_side"]).fillna(pd.Series(ids).map(seen_bat)).fillna("R").to_numpy()


def throw_hand_of(ids):
    return pd.Series(ids).map(hands["pitch_hand"]).fillna(pd.Series(ids).map(seen_throw)).fillna("R").to_numpy()


evaluable = coverage.query("category != 'excluded'")["game_pk"]
print(f"plate appearances 2022-2025: {len(pa):,} ({int(done.sum()):,} completed, {int((~done).sum()):,} interrupted)")
print("outcome shares:", (pa.loc[done, "outcome"].value_counts(normalize=True).sort_index()
                         .rename(dict(enumerate(OUTCOMES))).round(3).to_dict()))
print("games by coverage category:", coverage.pivot_table(index="category", columns="season", values="game_pk",
                                                          aggfunc="count", fill_value=0).to_dict("index"))
print("input checks passed: unique keys, seasons, 9-man lineups, 2 starters per game, outcome map")

plate appearances 2022-2025: 733,264 (731,531 completed, 1,733 interrupted)
outcome shares: {'K': 0.225, 'BB': 0.083, 'HBP': 0.012, '1B': 0.142, '2B': 0.043, '3B': 0.004, 'HR': 0.03, 'IPNH': 0.461}
games by coverage category: {'excluded': {2022: 0, 2023: 0, 2024: 1, 2025: 0, 2026: 1}, 'flagged': {2022: 7, 2023: 2, 2024: 4, 2025: 3, 2026: 1}, 'usable': {2022: 2423, 2023: 2428, 2024: 2425, 2025: 2427, 2026: 2428}}
input checks passed: unique keys, seasons, 9-man lineups, 2 starters per game, outcome map


## Player rates and log5

- **Season weights.** The latest season before the predicted one gets weight 1, and each earlier season `decay` times the next. A decay of 1.0 means equal weight per plate appearance. Fixing the latest weight at 1 means weights can't be rescaled to fake confidence.
- **Blending.** Each player's rate for each outcome is `(count + k x prior) / (n + k)`, then renormalized. The prior is the league rate for the same batter-side/pitcher-hand pairing, so a zero count can't make an outcome impossible. Players with no history get the league rates.
- **Blending strength.** `k` is estimated from each training window by method of moments: how much true player rates vary beyond sampling noise. The tuning step multiplies it by 0.5, 1 or 2.
- **Handedness.** Batters are split by pitcher hand and pitchers by batter side. Sides come from MLB profiles (current, not season-specific), then from the plays, then default to right-handed. Switch hitters bat from the side opposite the pitcher.
- **Log5** multiplies each outcome's batter and pitcher rates, divides by the league rate, and renormalizes. It is a formula, not a fitted model.

In [3]:
completed = pa[pa["outcome"].notna()].astype({"outcome": int})


def split_code(bat_side, pitch_hand):
    """Index of a batter-side/pitcher-hand split: 0 LL, 1 LR, 2 RL, 3 RR."""
    return (np.asarray(bat_side) == "R") * 2 + (np.asarray(pitch_hand) == "R")


def season_weights(target, decay):
    """Weight for each development season before `target`: the latest gets 1, each earlier one `decay` times the next."""
    return {s: decay ** (target - 1 - s) for s in DEV_SEASONS if s < target}


def weighted_counts(plays, by, weights):
    """Season-weighted outcome counts, one row per group and one column per outcome."""
    weighted = plays.assign(w=plays["season"].map(weights))
    return weighted.groupby(by + ["outcome"])["w"].sum().unstack(fill_value=0.0).reindex(columns=range(8), fill_value=0.0)


def estimate_blend_k(counts, league):
    """Blending strength per outcome, by method of moments: binomial variance over the spread of true player rates.

    Uses players with at least 300 weighted plate appearances. A larger k pulls a player harder toward the league.
    """
    n = counts.sum(axis=1)
    rates = counts[n >= 300].div(n[n >= 300], axis=0)
    noise = (league * (1 - league)) * (1 / n[n >= 300]).mean()
    true_var = rates.var(axis=0).to_numpy() - noise
    k = np.where(true_var > 0, league * (1 - league) / np.maximum(true_var, 1e-12) - 1, 5000.0)
    return np.clip(k, 10, 5000)


def blend(counts, prior, k):
    """(observed + k x prior) / (n + k) for each outcome, renormalized. A zero count still gets a positive rate."""
    n = counts.sum(axis=1, keepdims=True)
    rates = (counts + k * prior) / (n + k)
    return rates / rates.sum(axis=1, keepdims=True)


def build_profiles(target, decay, scale):
    """Batter, pitcher and league rates from development seasons before `target` only."""
    weights = season_weights(target, decay)
    window = completed[completed["season"].isin(list(weights))].assign(
        split=lambda d: split_code(d["bat_side"], d["pitch_hand"]))
    league_counts = weighted_counts(window, ["split"], weights).reindex(range(4)).to_numpy()
    league = league_counts / league_counts.sum(axis=1, keepdims=True)
    league_all = league_counts.sum(axis=0) / league_counts.sum()
    profile = {"target": target, "decay": decay, "scale": scale, "league": league, "league_all": league_all}
    # Batters are split by the pitcher's hand, pitchers by the batter's side. Each split is blended toward the league
    # rate for the same batter-side/pitcher-hand pairing (a switch hitter bats from the side opposite the pitcher).
    for role, player, opp, own in [("bat", "batter_id", "pitch_hand", "bat_side"), ("pit", "pitcher_id", "bat_side", "pitch_hand")]:
        counts = weighted_counts(window, [player, opp], weights)
        k = estimate_blend_k(weighted_counts(window, [player], weights), league_all) * scale
        own_side = window.groupby([player, opp])[own].agg(lambda s: s.mode().iat[0]).reindex(counts.index)
        opp_side = counts.index.get_level_values(opp)
        prior = league[split_code(*((own_side, opp_side) if role == "bat" else (opp_side, own_side)))]
        rates = blend(counts.to_numpy(), prior, k)
        index = pd.MultiIndex.from_arrays([counts.index.get_level_values(0).astype("int64"), opp_side])
        profile[role] = pd.DataFrame(rates, index=index)
        profile[f"{role}_n"] = pd.Series(counts.sum(axis=1).to_numpy(), index=index)
        profile[f"{role}_k"] = k
    return profile


def log5(batter, pitcher, league):
    """Multicategory log5: each outcome's odds scale by batter/league and pitcher/league; rows sum to 1."""
    raw = batter * pitcher / league
    return raw / raw.sum(axis=-1, keepdims=True)


def matchup_probs(profile, batter_ids, bat_sides, pitcher_ids, pitch_hands):
    """Outcome probabilities for each batter-pitcher pair. Players without history get the league split (newcomers)."""
    batter_ids, pitcher_ids = np.asarray(batter_ids, "int64"), np.asarray(pitcher_ids, "int64")
    league = profile["league"][split_code(bat_sides, pitch_hands)]
    b = profile["bat"].reindex(pd.MultiIndex.from_arrays([batter_ids, pitch_hands])).to_numpy(copy=True)
    p = profile["pit"].reindex(pd.MultiIndex.from_arrays([pitcher_ids, bat_sides])).to_numpy(copy=True)
    new_b, new_p = np.isnan(b[:, 0]), np.isnan(p[:, 0])
    b[new_b], p[new_p] = league[new_b], league[new_p]
    return log5(b, p, league), new_b, new_p


def side_against(bat_sides, pitch_hands):
    """Side a batter uses against a pitcher: switch hitters bat from the side opposite the pitcher's hand."""
    bat_sides, pitch_hands = np.asarray(bat_sides), np.asarray(pitch_hands)
    return np.where(bat_sides == "S", np.where(pitch_hands == "R", "L", "R"), bat_sides)


def pa_log_loss(profile, season):
    """Average log loss of the predicted outcome probabilities for one season's completed plate appearances."""
    plays = completed[completed["season"] == season]
    probs, _, _ = matchup_probs(profile, plays["batter_id"], plays["bat_side"], plays["pitcher_id"], plays["pitch_hand"])
    return float(-np.log(probs[np.arange(len(plays)), plays["outcome"].to_numpy()]).mean())


demo = build_profiles(TUNE_SEASON, 1.0, 1.0)
print("blending strength k (plate appearances), batters:", dict(zip(OUTCOMES, demo["bat_k"].round().astype(int).tolist())))
print("blending strength k (batters faced), pitchers:", dict(zip(OUTCOMES, demo["pit_k"].round().astype(int).tolist())))
print(f"2024 log loss from 2022-2023, equal weights: {pa_log_loss(demo, TUNE_SEASON):.4f}")

blending strength k (plate appearances), batters: {'K': 51, 'BB': 115, 'HBP': 233, '1B': 247, '2B': 1492, '3B': 598, 'HR': 205, 'IPNH': 81}
blending strength k (batters faced), pitchers: {'K': 75, 'BB': 194, 'HBP': 433, '1B': 376, '2B': 1045, '3B': 3017, 'HR': 758, 'IPNH': 192}
2024 log loss from 2022-2023, equal weights: 1.4706


## Runner movement

Each plate appearance has three recorded states: start, just before the deciding pitch, and after the play. The simulator splits them into two pieces, so each movement, run and out is counted once:

- **Whole-play table** (deciding pitch to end of play). Keyed by outcome, bases and outs, it gives where the batter and each runner ended up, including baserunning on the deciding pitch, such as a dropped third strike or a steal on ball four. Runners are matched by the base they started from because pinch runners can change IDs. Uses every completed play from seasons before the predicted one.
- **Before-the-pitch table** (start to deciding pitch). Keyed by bases and outs, it gives the combined effect of steals, wild pitches, pickoffs and balks, or nothing. Plate appearances cut off by an event run to the end of the play, so the inning ends and the same batter leads off the next inning. The extra-inning runner is placed by rule and is not an event. Uses 2023 onward.
- **Sparse cells.** A cell with fewer than 30 plays borrows the same outcome from other out counts. If a borrowed play now makes the third out, it ends the inning with no run credited. A cell with no data at all would use forced-advance rules. Plays whose third out ends the inning clear the bases, because where other runners stopped no longer matters.
- **Run accounting.** Runs on third-out plays use the official count. On a walk-off, the table keeps the runs that would naturally score and the game engine applies the walk-off rule.
- **Check.** The two pieces join exactly, and together they reproduce every 2022-2025 final score. The 8 games flagged because Savant pitch data disagreed with MLB are kept: their MLB runner states are consistent, and Savant data isn't used here.

In [4]:
BASE_NUM = {"1B": 1, "2B": 2, "3B": 3}
D_COLS = ["d0", "d1", "d2", "d3"]   # destination of the batter (0) and of the runners on first, second and third
last_play = pa.groupby("game_pk")["at_bat_index"].transform("max") == pa["at_bat_index"]


def play_records():
    """Whole-play results of completed plate appearances, from the state just before the deciding pitch to after it.

    Destinations: 0 out, 1-3 a base, 4 scored, -1 nobody started there. Runners are matched by the base they started
    from (a pinch runner can carry a new ID), and a runner with no recorded leg stays where he was.
    """
    lg = legs[legs["phase"].isin(["batter_outcome", "same_pitch_baserunning"])].sort_values(
        ["game_pk", "at_bat_index", "play_index", "leg"])
    key = ["game_pk", "at_bat_index", "runner_id"]
    first, last = lg.drop_duplicates(key, keep="first"), lg.drop_duplicates(key, keep="last").set_index(key)
    runners = first[key + ["is_batter", "start_base"]].join(last[["end_base", "is_out"]], on=key)
    runners["origin"] = np.where(runners["is_batter"], 0, runners["start_base"].map(BASE_NUM))
    runners["dest"] = np.where(runners["is_out"].eq(True), 0,
                               np.where(runners["end_base"].eq("score"), 4, runners["end_base"].map(BASE_NUM)))
    clash = runners.duplicated(["game_pk", "at_bat_index", "origin"], keep=False) | runners["origin"].isna()
    dest = runners[~clash].set_index(["game_pk", "at_bat_index", "origin"])["dest"].unstack()
    dest = dest.reindex(columns=[0, 1, 2, 3]).set_axis(D_COLS, axis=1)
    rec = pa[pa["outcome"].notna()].join(dest, on=["game_pk", "at_bat_index"])
    rec["bad_legs"] = rec.set_index(["game_pk", "at_bat_index"]).index.isin(
        runners.loc[clash, ["game_pk", "at_bat_index"]].apply(tuple, axis=1))
    for b in (1, 2, 3):
        on_base = (rec["pre_mask"].to_numpy() >> (b - 1)) & 1 == 1
        rec[f"d{b}"] = np.where(on_base, rec[f"d{b}"].fillna(b), np.where(rec[f"d{b}"].isna(), -1, 99))  # 99: leg from an empty base
    rec["outs"] = rec["post_outs"] - rec["pre_outs"]
    delta = rec["post_bat_score"] - rec["pre_bat_score"]
    d = rec[D_COLS].to_numpy()
    scored, outs_seen = (d == 4).sum(axis=1), (d == 0).sum(axis=1)
    end_mask = sum(((d == b).any(axis=1)).astype(int) << (b - 1) for b in (1, 2, 3))
    crowded = np.max([(d == b).sum(axis=1) for b in (1, 2, 3)], axis=0)   # runners ending on the same base
    walk_off = last_play.loc[rec.index] & (rec["half"] == "bottom") & (scored > delta)
    # A run on the third out counts only if it scored before that out, so 3-out plays take the official run count.
    rec["runs"] = np.where(rec["post_outs"] == 3, delta, scored)
    rec["problem"] = np.select(
        [rec["bad_legs"], np.isnan(d).any(axis=1) | (d == 99).any(axis=1), outs_seen != rec["outs"],
         (rec["post_outs"] < 3) & ((end_mask != rec["post_mask"]) | (crowded > 1)),
         (rec["post_outs"] < 3) & (scored != delta) & ~walk_off],
        ["runner listed twice", "missing or impossible leg", "outs disagree", "bases disagree", "runs disagree"], "")
    d = np.nan_to_num(d, nan=-1).astype(int)
    # A third out ends the inning, so where the other runners stopped no longer matters: they leave the bases
    d[(rec["post_outs"].to_numpy() == 3)[:, None] & (d >= 1) & (d <= 3)] = -1
    rec[D_COLS] = d
    return rec


def event_records():
    """Runner changes before the deciding pitch: start of the plate appearance to just before the deciding pitch.

    Plate appearances cut off by an event (usually an inning-ending caught stealing) run to the end of the play, so
    the inning ends without the batter finishing. Runners are matched by player ID; a pinch runner who appears
    mid-plate-appearance takes over the nearest vacated base behind him.
    """
    rec = pa.copy()
    # The extra-inning runner is placed by rule, not by an event, so that runner counts as already on second
    placed = events[events["event_type"] == "runner_placed"].drop_duplicates(["game_pk", "at_bat_index"])
    placed = placed.set_index(["game_pk", "at_bat_index"])["after_on_2b"]
    has_placed = rec.set_index(["game_pk", "at_bat_index"]).index.isin(placed.index)
    rec.loc[has_placed, "start_on_2b"] = placed.reindex(rec.loc[has_placed, ["game_pk", "at_bat_index"]].apply(tuple, axis=1)).to_numpy()
    cut = rec["is_interrupted"].to_numpy()
    start_ids = rec[[f"start_on_{b}b" for b in (1, 2, 3)]].to_numpy("float64", na_value=np.nan)
    end_ids = np.where(cut[:, None], rec[[f"post_on_{b}b" for b in (1, 2, 3)]].to_numpy("float64", na_value=np.nan),
                       rec[[f"pre_on_{b}b" for b in (1, 2, 3)]].to_numpy("float64", na_value=np.nan))
    start_outs = rec["start_outs"].to_numpy()
    outs = np.where(cut, rec["post_outs"], rec["pre_outs"]) - start_outs
    runs = np.where(cut, rec["post_bat_score"], rec["pre_bat_score"]) - rec["start_bat_score"].to_numpy()
    dest = np.full((len(rec), 4), -1.0)
    for b in range(3):
        on = ~np.isnan(start_ids[:, b])
        dest[on, b + 1] = np.nan                     # unresolved until matched below
        for e in range(3):
            dest[on & (start_ids[:, b] == end_ids[:, e]), b + 1] = e + 1
    newcomer = ~np.isnan(end_ids) & ~(end_ids[:, :, None] == start_ids[:, None, :]).any(axis=2)
    for i in np.flatnonzero(newcomer.any(axis=1)):   # pinch runners (a few hundred rows)
        for e in np.flatnonzero(newcomer[i]):
            open_origins = [o for o in (3, 2, 1) if o <= e + 1 and np.isnan(dest[i, o])]
            if open_origins:
                dest[i, open_origins[0]] = e + 1
    missing = np.isnan(dest)
    problem = np.where(missing.sum(axis=1) != outs + runs, "runner count disagrees", "")
    # Vanished runners scored or were put out; lead runners are credited with the runs first.
    to_score = runs.copy()
    for b in (3, 2, 1):
        m = missing[:, b]
        dest[m, b] = np.where(to_score[m] > 0, 4, 0)
        to_score[m] -= to_score[m] > 0
    dest = np.nan_to_num(dest, nan=-1).astype(int)
    dest[(start_outs + outs == 3)[:, None] & (dest >= 1) & (dest <= 3)] = -1   # inning over: bases clear
    rec[D_COLS] = dest
    rec["outs"], rec["runs"] = outs, runs
    rec["start_mask"] = sum((~np.isnan(start_ids[:, b])).astype(int) << b for b in range(3))
    rec["problem"] = np.where(cut & (rec["post_outs"] < 3) & (runs == 0), "cut off without an inning or game end", problem)
    rec["pre_outs_from"] = start_outs
    return rec[rec["start_mask"] > 0]


def forced_outcome(cat, mask, outs):
    """Deterministic fallback for a play never seen: runners move only as far as the hit or a force requires."""
    on = [(mask >> b) & 1 for b in range(3)]
    if cat in (K_, IPNH_):
        d = [0] + [b + 1 if on[b] else -1 for b in range(3)]
    elif cat in (BB_, HBP_):
        d = [1, 2 if on[0] else -1, (3 if on[0] else 2) if on[1] else -1, (4 if on[0] and on[1] else 3) if on[2] else -1]
    else:
        bases = {S1_: 1, S2_: 2, S3_: 3, HR_: 4}[cat]
        d = [bases] + [min(b + 1 + bases, 4) if on[b] else -1 for b in range(3)]
    out_add = d.count(0)
    runs = 0 if outs + out_add >= 3 else d.count(4)
    return (*d, out_add, runs)


def pack(table, n_keys):
    """Turn {key: [(d0, d1, d2, d3, outs, runs, prob), ...]} into padded arrays for vectorized sampling."""
    width = max(len(v) for v in table.values())
    out = {"dest": np.full((n_keys, width, 4), -1, np.int8), "outs": np.zeros((n_keys, width), np.int8),
           "runs": np.zeros((n_keys, width), np.int8), "cum": np.ones((n_keys, width))}
    for key, rows in table.items():
        rows = np.array(rows, dtype=float)
        n = len(rows)
        out["dest"][key, :n] = rows[:, :4]
        out["outs"][key, :n], out["runs"][key, :n] = rows[:, 4], rows[:, 5]
        out["cum"][key, :n] = np.cumsum(rows[:, 6]) / rows[:, 6].sum()
        out["cum"][key, n - 1:] = 1.0
    return out


def build_table(records, key_of, keys, pool_of, min_cell, empty_rule):
    """Count outcomes per key; a sparse key borrows the same play at any number of outs, else a deterministic rule."""
    cols = D_COLS + ["outs", "runs"]
    counts = records.assign(key=key_of(records)).groupby(["key"] + cols).size()
    table, sources = {}, {"own": 0, "pooled": 0, "rule": 0}
    for key, (cat, mask, outs) in keys.items():
        rows = counts.loc[key] if key in counts.index.get_level_values(0) else pd.Series(dtype=float)
        if rows.sum() >= min_cell:
            table[key] = [(*idx, w) for idx, w in rows.items()]
            sources["own"] += 1
            continue
        pool = pool_of(records, cat, mask)
        pool = pool[pool["outs"] <= 3 - outs].copy()
        third = pool["outs"] + outs >= 3
        pool.loc[third, "runs"] = 0   # borrowed plays that now make the third out end the inning with no run credited
        for col in D_COLS:
            pool.loc[third & pool[col].between(1, 3), col] = -1
        if len(pool) >= min_cell or (len(pool) and empty_rule is None):
            table[key] = [(*idx, w) for idx, w in pool.groupby(cols).size().items()]
            sources["pooled"] += 1
        else:
            table[key] = [(*empty_rule(cat, mask, outs), 1.0)]
            sources["rule"] += 1
    return pack(table, max(keys) + 1), sources


PLAY_KEYS = {cat * 24 + mask * 3 + outs: (cat, mask, outs) for cat in range(8) for mask in range(8) for outs in range(3)}
EVENT_KEYS = {mask * 3 + outs: (None, mask, outs) for mask in range(8) for outs in range(3)}
NO_EVENT = lambda cat, mask, outs: (-1, *[b + 1 if (mask >> b) & 1 else -1 for b in range(3)], 0, 0)


def build_runner_tables(target, plays, moves):
    """Whole-play and before-the-pitch tables from seasons before `target` (events from 2023 on)."""
    window = plays[(plays["season"] < target) & (plays["problem"] == "")]
    ev = moves[(moves["season"] < target) & (moves["season"] >= EVENT_FIRST_SEASON) & (moves["problem"] == "")]
    # Every plate appearance with runners on contributes a row, so "nothing happened" is the most common outcome
    play_table, play_src = build_table(
        window, lambda r: r["outcome"].astype(int) * 24 + r["pre_mask"] * 3 + r["pre_outs"], PLAY_KEYS,
        lambda r, cat, mask: r[(r["outcome"] == cat) & (r["pre_mask"] == mask)], MIN_PLAY_CELL, forced_outcome)
    event_table, event_src = build_table(
        ev, lambda r: r["start_mask"] * 3 + r["pre_outs_from"], {k: v for k, v in EVENT_KEYS.items() if v[1] > 0},
        lambda r, cat, mask: r[r["start_mask"] == mask], MIN_PLAY_CELL, NO_EVENT)
    for key in (k for k, v in EVENT_KEYS.items() if v[1] == 0):   # bases empty: nothing can happen before the pitch
        event_table["cum"][key] = 1.0
        event_table["dest"][key], event_table["outs"][key], event_table["runs"][key] = -1, 0, 0
    return {"play": play_table, "event": event_table, "play_sources": play_src, "event_sources": event_src}


def draw(table, keys, u):
    """Index of the sampled outcome for each row's key, from uniform draws u."""
    return (table["cum"][keys] < u[:, None]).sum(axis=1)


def move_runners(bases, batter, dests, check=False):
    """New base occupants (batting slots, -1 empty) after each origin's runner goes to its destination."""
    occupants = np.column_stack([batter, bases])
    new = np.full(bases.shape, -1, bases.dtype)
    for b in range(3):
        landing = dests == b + 1
        if check and ((landing.sum(axis=1) > 1).any() or (landing & (occupants < 0)).any()):
            raise AssertionError("illegal runner move: two runners on one base, or a runner from an empty base")
        rows, origin = np.nonzero(landing)
        new[rows, b] = occupants[rows, origin]
    return new


plays_all, moves_all = play_records(), event_records()
summary = pd.DataFrame({
    "records": [len(plays_all), len(moves_all)],
    "used": [int((plays_all["problem"] == "").sum()), int((moves_all["problem"] == "").sum())],
    "excluded": [int((plays_all["problem"] != "").sum()), int((moves_all["problem"] != "").sum())]},
    index=["whole plays (completed plate appearances)", "before-the-pitch changes (runners on at the start)"])
display(summary)
print("exclusion reasons:", {**plays_all["problem"][plays_all["problem"] != ""].value_counts().to_dict(),
                             **moves_all["problem"][moves_all["problem"] != ""].value_counts().to_dict()})
# Counted once: the before-the-pitch segment ends exactly where the whole play starts, and together the two
# segments reproduce every official final score.
segments_tile = (moves_all.loc[~moves_all["is_interrupted"], "pre_mask"] ==
                 sum(((moves_all.loc[~moves_all["is_interrupted"], D_COLS[1:]].to_numpy() == b).any(axis=1)).astype(int) << (b - 1)
                     for b in (1, 2, 3))).mean()
seg_runs = (pa["post_bat_score"] - pa["start_bat_score"]).groupby([pa["game_pk"], pa["batting_team_id"]]).sum()
official = dev_results.merge(games[["game_pk", "home_team_id", "away_team_id"]], on="game_pk").query("status != 'Cancelled'")
rebuilt_home = seg_runs.reindex(pd.MultiIndex.from_frame(official[["game_pk", "home_team_id"]])).to_numpy()
rebuilt_away = seg_runs.reindex(pd.MultiIndex.from_frame(official[["game_pk", "away_team_id"]])).to_numpy()
matches = (rebuilt_home == official["home_score"].to_numpy()) & (rebuilt_away == official["away_score"].to_numpy())
print(f"before-the-pitch end state equals the play's starting bases: {segments_tile:.4%}; "
      f"segment runs reproduce the official score in {int(matches.sum())} of {len(official)} completed games")

,records,used,excluded
whole plays (completed plate appearances),731531,731531,0
before-the-pitch changes (runners on at the start),321563,321560,3


exclusion reasons: {'cut off without an inning or game end': 3}
before-the-pitch end state equals the play's starting bases: 100.0000%; segment runs reproduce the official score in 9719 of 9719 completed games


## Pitching

- **Starter removal** is a logistic regression fitted with SciPy, separate from log5. It has one row per boundary between plays while the starter is still in, with the label "a different pitcher threw the next play". The inputs are batters faced, simulated pitches, runs allowed, inning, outs, the start of an inning, the score margin, and the starter's usual length. Usual length is the mean batters faced in earlier starts, blended with 3 league-average starts. It lets pitchers used as openers leave early, but the bulk innings after an opener come from the pooled bullpen.
- **Pitch counts** are drawn from the historical pitches-per-plate-appearance distribution for the drawn outcome. This is an approximation; pitches aren't simulated one by one.
- **Pooled bullpen.** After the starter leaves, each team pitches with one profile: a usage-weighted mix of log5 probabilities against every eligible reliever. A reliever's weight is relief batters faced per listed team-game in earlier seasons, blended with 20 league-average games. This measures usage, not availability. Availability comes only from the pregame bullpen list.
- **Eligibility.** The baseline uses `membership = member`. The sensitivity run also includes `uncertain` (same-day moves and trades). `not_member` and the day's starting pitchers are never included. An empty pool falls back to a league-average reliever.
- **Not represented:** individual reliever choice, roles, rest, the three-batter minimum, and position players pitching.
- **Attribution uncertainty.** Starting-pitcher disagreements in 9 flagged games and the boxscore's batters-faced offsets (41 offsetting pairs within team-games) affect who is credited with a play. They don't affect runner states. Predictions use the box score's starting pitcher.

In [5]:
TEAM_KEYS = ["game_pk", "fielding_team_id"]


def removal_features(bf, pc, runs, inning, outs, new_inning, margin, usual):
    """Starter-removal predictors at a plate-appearance boundary. Used both for fitting and inside the simulator."""
    bf, usual = np.asarray(bf, float) / 10, np.asarray(usual, float) / 10
    pc = np.asarray(pc, float) / 100
    return np.column_stack([np.ones_like(bf), bf, bf ** 2, pc, pc ** 2, np.minimum(runs, 10), np.asarray(inning) / 9,
                            new_inning, outs, np.clip(margin, -10, 10) / 5, usual, bf - usual])


def starts_table(seasons):
    """One row per start: the first pitcher of each team-game and the completed plays faced before the first change."""
    d = pa[pa["season"].isin(seasons)]
    starter = d.groupby(TEAM_KEYS, sort=False)["pitcher_id"].transform("first")
    still_in = (d["pitcher_id"] == starter).astype(int).groupby([d["game_pk"], d["fielding_team_id"]]).cummin().astype(bool)
    return d.assign(starter=starter, still_in=still_in), (d[still_in & ~d["is_interrupted"]]
            .groupby(TEAM_KEYS + ["game_order"]).agg(pitcher_id=("pitcher_id", "first"), bf=("pitcher_id", "size"))
            .reset_index().sort_values("game_order"))


def usual_length(starts, league_mean):
    """Each start's 'usual length': mean batters faced in the pitcher's earlier starts, blended with league starts."""
    g = starts.groupby("pitcher_id")["bf"]
    prior_sum, prior_n = g.cumsum() - starts["bf"], g.cumcount()
    return (prior_sum + USUAL_LENGTH_PRIOR * league_mean) / (prior_n + USUAL_LENGTH_PRIOR)


def fit_removal_model(seasons):
    """Logistic regression for 'the starter is replaced before this plate appearance', fitted with SciPy.

    One row per boundary between two plays by the same fielding team while the starter is still in. A game that ends
    with the starter still pitching has no next play, so its end is never mistaken for a removal. Pitches, runs and
    batters faced come from the plays (the boxscore's batters faced differs by one in 41 offsetting pairs of
    appearances, which only shifts a play's credit between two pitchers of the same team).
    """
    d, starts = starts_table(seasons)
    league_mean = starts["bf"].mean()
    starts["usual"] = usual_length(starts, league_mean)
    keys = [d["game_pk"], d["fielding_team_id"]]
    bf = (d["still_in"] & ~d["is_interrupted"]).astype(int).groupby(keys).cumsum()
    pc = d["pitches"].where(d["still_in"], 0).groupby(keys).cumsum()
    nxt = d.groupby(TEAM_KEYS, sort=False)[["pitcher_id", "inning", "start_outs", "start_bat_score", "start_fld_score"]].shift(-1)
    rows = d["still_in"] & nxt["pitcher_id"].notna()
    usual = d[TEAM_KEYS].merge(starts[TEAM_KEYS + ["usual"]], on=TEAM_KEYS, how="left")["usual"].to_numpy()
    X = removal_features(bf[rows], pc[rows], nxt.loc[rows, "start_bat_score"], nxt.loc[rows, "inning"],
                         nxt.loc[rows, "start_outs"], (nxt["inning"] != d["inning"])[rows],
                         (nxt["start_fld_score"] - nxt["start_bat_score"])[rows], usual[rows.to_numpy()])
    y = (nxt.loc[rows, "pitcher_id"] != d.loc[rows, "starter"]).to_numpy(float)

    def loss(beta):   # mean negative log-likelihood with a small ridge penalty (intercept unpenalized)
        z = X @ beta
        penalty = 1e-4 * np.r_[0.0, beta[1:]]
        return (np.logaddexp(0, z) - y * z).mean() + (penalty * beta).sum(), X.T @ (special.expit(z) - y) / len(y) + 2 * penalty

    fit = optimize.minimize(loss, np.zeros(X.shape[1]), jac=True, method="L-BFGS-B", options={"maxiter": 2000})
    if not fit.success:
        raise RuntimeError(f"starter-removal fit did not converge: {fit.message}")
    usual_by_pitcher = starts.groupby("pitcher_id")["bf"].agg(["sum", "count"])
    usual_final = (usual_by_pitcher["sum"] + USUAL_LENGTH_PRIOR * league_mean) / (usual_by_pitcher["count"] + USUAL_LENGTH_PRIOR)
    return {"beta": fit.x, "rows": len(y), "removal_rate": y.mean(), "league_start_bf": league_mean,
            "usual": usual_final, "starts": starts}


def pitch_draws(seasons):
    """Pitches per plate appearance by outcome, as sorted arrays to sample from (an approximation of pitch counts)."""
    d = completed[completed["season"].isin(seasons)]
    values = [np.sort(d.loc[d["outcome"] == c, "pitches"].to_numpy()) for c in range(8)]
    return {"values": np.concatenate(values), "start": np.cumsum([0] + [len(v) for v in values[:-1]]),
            "count": np.array([len(v) for v in values])}


def relief_shares(seasons):
    """Each pitcher's usual share of bullpen work: relief batters faced per team-game listed, blended toward the league.

    This measures usage when available, not availability. Who is available comes from the pregame bullpen list.
    """
    relief = appearances[appearances["season"].isin(seasons) & ~appearances["is_starter"]].groupby("pitcher_id")["batters_faced"].sum()
    listed = bullpen[bullpen["season"].isin(seasons) & bullpen["membership"].isin(["member", "uncertain"])
                     & ~bullpen["is_starting_pitcher"]].groupby("player_id").size()
    per_listing = relief.sum() / listed.sum()
    listed = listed.reindex(listed.index.union(relief.index), fill_value=0)
    share = (relief.reindex(listed.index, fill_value=0) + RELIEF_SHARE_PRIOR * per_listing) / (listed + RELIEF_SHARE_PRIOR)
    return share, per_listing


def game_inputs(game_pks, profile, removal, shares, include_uncertain):
    """Per-game matchup probabilities: each team's nine batters against the opposing starter and the pooled bullpen.

    The bullpen is one profile: a usage-weighted mix of log5 probabilities against every eligible reliever. Eligible
    means on the pregame list (member, plus uncertain only in the sensitivity run), never today's starting pitcher.
    """
    share, per_listing = shares
    lu = lineups[lineups["game_pk"].isin(game_pks)].sort_values(["game_pk", "side", "batting_slot"])
    st = starters[starters["game_pk"].isin(game_pks)]
    opp = {"away": "home", "home": "away"}
    lu = lu.assign(opp_side=lu["side"].map(opp)).merge(
        st[["game_pk", "side", "player_id", "team_id"]].rename(columns={"side": "opp_side", "player_id": "pitcher_id", "team_id": "opp_team"}),
        on=["game_pk", "opp_side"])
    lu["bat_hand"] = bat_side_of(lu["player_id"])
    lu["throw"] = throw_hand_of(lu["pitcher_id"])
    p_start, new_b, new_sp = matchup_probs(profile, lu["player_id"], side_against(lu["bat_hand"], lu["throw"]), lu["pitcher_id"], lu["throw"])
    allowed = ["member", "uncertain"] if include_uncertain else ["member"]
    pen = bullpen[bullpen["game_pk"].isin(game_pks) & bullpen["membership"].isin(allowed) & ~bullpen["is_starting_pitcher"]]
    pen = pen.merge(st[["game_pk", "player_id"]].rename(columns={"player_id": "sp"}), on="game_pk")
    pen = pen[pen["player_id"] != pen["sp"]].drop_duplicates(["game_pk", "team_id", "player_id"])
    pool_size = pen.groupby(["game_pk", "team_id"]).size()
    empty = lu[["game_pk", "opp_team"]].drop_duplicates()
    empty = empty[~pd.MultiIndex.from_frame(empty).isin(pool_size.index)]
    # An empty pool falls back to a league-average reliever (pitcher ID -1 has no history, so log5 uses the league)
    pen = pd.concat([pen[["game_pk", "team_id", "player_id"]],
                     empty.rename(columns={"opp_team": "team_id"}).assign(player_id=-1)], ignore_index=True)
    pen["weight"] = pen["player_id"].map(share).fillna(per_listing)
    pen["throw"] = throw_hand_of(pen["player_id"])
    pairs = lu.reset_index(drop=True).reset_index()[["index", "game_pk", "opp_team", "player_id", "bat_hand"]].merge(
        pen.rename(columns={"team_id": "opp_team", "player_id": "pitcher_id"}), on=["game_pk", "opp_team"])
    probs, _, new_p = matchup_probs(profile, pairs["player_id"], side_against(pairs["bat_hand"], pairs["throw"]), pairs["pitcher_id"], pairs["throw"])
    weighted = pd.DataFrame(probs * pairs["weight"].to_numpy()[:, None]).groupby(pairs["index"]).sum()
    p_pen = (weighted.to_numpy() / pairs.groupby("index")["weight"].sum().to_numpy()[:, None])
    lu = lu.reset_index(drop=True)
    usual = lu["pitcher_id"].map(removal["usual"]).fillna(removal["league_start_bf"]).to_numpy()
    inputs = {}
    side_index = {"away": 0, "home": 1}
    for game_pk, rows in lu.groupby("game_pk").groups.items():
        rows = np.asarray(rows)
        team = lu.loc[rows, "side"].map(side_index).to_numpy()
        order = np.lexsort((lu.loc[rows, "batting_slot"].to_numpy(), team))
        rows = rows[order]
        inputs[game_pk] = {"p_start": p_start[rows].reshape(2, 9, 8), "p_pen": p_pen[rows].reshape(2, 9, 8),
                           # usual length of [away starter, home starter]; home's starter faces the away batters (rows 0-8)
                           "usual": np.array([usual[rows[9]], usual[rows[0]]])}
    info = pd.DataFrame({"batters_without_history": pd.Series(new_b, index=lu.index).groupby(lu["game_pk"]).sum(),
                         "starters_without_history": pd.Series(new_sp, index=lu.index).groupby([lu["game_pk"], lu["pitcher_id"]]).first()
                         .groupby(level=0).sum(),
                         "starters_without_starts": lu.drop_duplicates(["game_pk", "pitcher_id"]).assign(
                             none=lambda d: ~d["pitcher_id"].isin(removal["usual"].index)).groupby("game_pk")["none"].sum(),
                         "bullpen_size": pen[pen["player_id"] != -1].groupby("game_pk").size(),
                         "empty_bullpens": pen[pen["player_id"] == -1].groupby("game_pk").size(),
                         "uncertain_listed": bullpen[bullpen["game_pk"].isin(game_pks) & (bullpen["membership"] == "uncertain")]
                         .groupby("game_pk").size()}).reindex(list(inputs)).fillna(0).astype(int)
    return inputs, info


removal_demo = fit_removal_model([2022, 2023])
print(f"starter-removal model fitted on {removal_demo['rows']:,} boundaries from 2022-2023; "
      f"removal rate per boundary {removal_demo['removal_rate']:.3f}; league mean start {removal_demo['league_start_bf']:.1f} batters")

starter-removal model fitted on 213,301 boundaries from 2022-2023; removal rate per boundary 0.045; league mean start 21.9 batters


## Game engine

All 1,000 copies of a game advance together with NumPy arrays: one plate appearance per step for every unfinished copy. Each step:

1. Checks for a starter change (between plate appearances only).
2. Places the extra-inning runner from the 10th inning on.
3. Draws events before the deciding pitch.
4. Draws the plate appearance and the whole-play result.

Base runners are stored as batting slots, so identities carry through every move. The engine applies the rules: universal designated hitter (the nine starters bat all game; two-way players simply stay in the lineup), the bottom of the 9th skipped when the home team leads, walk-offs (only the winning run counts unless it's a home run), and no ties. A game still going after 60 innings raises an error instead of being given a winner. Each game has its own generator seeded from `(SEED, game_pk)`, so results don't depend on run order, and both bullpen variants use the same random numbers.

In [6]:
def base_mask(bases):
    return (bases[:, 0] >= 0) + 2 * (bases[:, 1] >= 0) + 4 * (bases[:, 2] >= 0)


def new_state(n):
    """Starting state for n copies of a game. Team 0 is away (bats in the top half), team 1 is home."""
    return {"inning": np.ones(n, np.int16), "half": np.zeros(n, np.int8), "outs": np.zeros(n, np.int8),
            "bases": np.full((n, 3), -1, np.int8),          # batting slot (0-8) of the runner on each base, -1 empty
            "score": np.zeros((n, 2), np.int16), "bat": np.zeros((n, 2), np.int8),   # next batting slot per team
            "starter_in": np.ones((n, 2), bool), "bf": np.zeros((n, 2), np.int16), "pitches": np.zeros((n, 2), np.int16),
            "new_half": np.ones(n, bool), "over": np.zeros(n, bool)}


def walk_off(s, rows, is_homer):
    """End the game when the home team takes the lead in the 9th inning or later. Without a home run, only the
    winning run counts; on a home run every run counts."""
    won = (s["half"][rows] == 1) & (s["inning"][rows] >= 9) & (s["score"][rows, 1] > s["score"][rows, 0])
    capped = rows[won & ~is_homer]
    s["score"][capped, 1] = s["score"][capped, 0] + 1
    s["over"][rows[won]] = True


def simulate_game(inputs, model, n_sims, rng, check=False, state=None, max_steps=None):
    """Simulate one game n_sims times, one plate appearance per step for every unfinished copy.

    Pitching changes happen only between plate appearances. Once the starter leaves, the team's pooled bullpen pitches
    the rest of the game; individual relievers, their rest and the three-batter minimum are not represented.
    """
    s = state if state is not None else new_state(n_sims)
    cum_start, cum_pen = np.cumsum(inputs["p_start"], axis=-1), np.cumsum(inputs["p_pen"], axis=-1)
    play, event, beta, pitch = model["play"], model["event"], model["beta"], model["pitches"]
    steps = 0
    while not s["over"].all() and (max_steps is None or steps < max_steps):
        steps += 1
        i = np.flatnonzero(~s["over"])
        if s["inning"][i].max() > MAX_INNINGS or steps > 100 * MAX_INNINGS:
            raise RuntimeError(f"termination guard: {len(i)} simulations unfinished after {steps} steps "
                               f"(up to inning {int(s['inning'][i].max())}); no winner is forced")
        t = s["half"][i].astype(np.intp)       # batting team
        f = 1 - t                               # fielding team
        # 1. Pitching change before this plate appearance (the starter always faces the first batter)
        cand = s["starter_in"][i, f] & (s["bf"][i, f] > 0)
        if cand.any():
            j, tj, fj = i[cand], t[cand], f[cand]
            x = removal_features(s["bf"][j, fj], s["pitches"][j, fj], s["score"][j, tj], s["inning"][j], s["outs"][j],
                                 s["new_half"][j], s["score"][j, fj] - s["score"][j, tj], inputs["usual"][fj])
            leaves = rng.random(len(j)) < special.expit(x @ beta)
            s["starter_in"][j[leaves], fj[leaves]] = False
        # 2. From the 10th inning, the batter before this half's leadoff hitter starts on second
        place = s["new_half"][i] & (s["inning"][i] >= 10)
        s["bases"][i[place], 1] = (s["bat"][i[place], t[place]] - 1) % 9
        s["new_half"][i] = False
        # 3. Steals, wild pitches, pickoffs and similar events before the deciding pitch
        keys = base_mask(s["bases"][i]) * 3 + s["outs"][i]
        k = draw(event, keys, rng.random(len(i)))
        s["bases"][i] = move_runners(s["bases"][i], np.full(len(i), -1, np.int8), event["dest"][keys, k], check)
        s["outs"][i] += event["outs"][keys, k]
        s["score"][i, t] += event["runs"][keys, k]
        walk_off(s, i, np.zeros(len(i), bool))
        # 4. The plate appearance, unless an event ended the inning (the same batter then leads off next inning)
        go = (s["outs"][i] < 3) & ~s["over"][i]
        j, tj, fj = i[go], t[go], f[go]
        slot = s["bat"][j, tj].astype(np.intp)
        starter = s["starter_in"][j, fj]
        cum = np.where(starter[:, None], cum_start[tj, slot], cum_pen[tj, slot])
        cat = np.minimum((cum < rng.random(len(j))[:, None]).sum(axis=1), 7)
        u = rng.random(len(j))
        thrown = pitch["values"][pitch["start"][cat] + (u * pitch["count"][cat]).astype(int)]
        s["bf"][j[starter], fj[starter]] += 1
        s["pitches"][j[starter], fj[starter]] += thrown[starter]
        keys = cat * 24 + base_mask(s["bases"][j]) * 3 + s["outs"][j]
        k = draw(play, keys, rng.random(len(j)))
        s["bases"][j] = move_runners(s["bases"][j], slot.astype(np.int8), play["dest"][keys, k], check)
        s["outs"][j] += play["outs"][keys, k]
        s["score"][j, tj] += play["runs"][keys, k]
        s["bat"][j, tj] = (slot + 1) % 9
        walk_off(s, j, cat == HR_)
        # 5. Three outs end the half-inning. The bottom of the 9th is skipped when home leads; a tie continues.
        e = i[(s["outs"][i] >= 3) & ~s["over"][i]]
        lead = s["score"][e, 1] - s["score"][e, 0]
        late = s["inning"][e] >= 9
        ends = np.where(s["half"][e] == 0, late & (lead > 0), late & (lead != 0))
        s["over"][e[ends]] = True
        e = e[~ends]
        s["inning"][e] += s["half"][e]          # after the bottom half, move to the next inning
        s["half"][e] = 1 - s["half"][e]
        s["outs"][e], s["bases"][e], s["new_half"][e] = 0, -1, True
        if check:
            live = ~s["over"]
            if (s["outs"][live] > 2).any() or (s["score"] < 0).any():
                raise AssertionError("illegal state: an unfinished half-inning has 3 outs, or a score is negative")
            occupied = s["bases"][live]
            if any(((occupied == slot_) .sum(axis=1) > 1).any() for slot_ in range(9)):
                raise AssertionError("illegal state: one batter on two bases")
    return {"away": s["score"][:, 0].copy(), "home": s["score"][:, 1].copy(), "innings": s["inning"].copy(),
            "starter_bf": s["bf"].copy(), "state": s, "steps": steps}


def make_model(target, decay, scale):
    """Everything estimated from development seasons before `target`, with one setting of weights and blending."""
    seasons = [s for s in DEV_SEASONS if s < target]
    tables = build_runner_tables(target, plays_all, moves_all)
    removal = fit_removal_model(seasons)
    return {"target": target, "profile": build_profiles(target, decay, scale), "play": tables["play"],
            "event": tables["event"], "sources": (tables["play_sources"], tables["event_sources"]),
            "removal": removal, "beta": removal["beta"], "pitches": pitch_draws(seasons), "shares": relief_shares(seasons)}


def simulate_games(game_pks, model, variant, n_sims=N_SIMS, check=False):
    """Simulate each game n_sims times. Every game has its own seeded generator, so results don't depend on order,
    and the baseline and sensitivity runs share random numbers game by game."""
    inputs, info = game_inputs(game_pks, model["profile"], model["removal"], model["shares"], variant == "sensitivity")
    rows, totals = [], np.zeros(40, int)
    for game_pk in inputs:
        rng = np.random.default_rng([SEED, int(game_pk)])
        sim = simulate_game(inputs[game_pk], model, n_sims, rng, check=check)
        wins = int((sim["home"] > sim["away"]).sum())
        rows.append({"game_pk": int(game_pk), "home_wins": wins, "n_sims": n_sims,
                     "p_home": (wins + SMOOTHING) / (n_sims + 2 * SMOOTHING),
                     "home_runs_mean": sim["home"].mean(), "away_runs_mean": sim["away"].mean(),
                     "total_runs_sd": (sim["home"] + sim["away"]).std(), "p_extra_innings": (sim["innings"] > 9).mean(),
                     "home_starter_bf_mean": sim["starter_bf"][:, 1].mean(), "away_starter_bf_mean": sim["starter_bf"][:, 0].mean()})
        totals += np.bincount(np.minimum(sim["home"] + sim["away"], 39), minlength=40)
    return pd.DataFrame(rows).merge(info, left_on="game_pk", right_index=True), totals

## Correctness checks and timing pilot

These tests check that the code follows the rules. They say nothing about predictive accuracy:

- **Probabilities and rates:** valid probabilities, the log5 identity, positive rates for zero counts, and season-weight scaling.
- **Runner tables:** legal moves, at most 3 outs, and no run when the batter is retired for the third out.
- **Scripted game situations:** a forced walk, a double play, an inning-ending caught stealing, the extra-inning runner, walk-offs with and without a home run, the skipped and played bottom of the 9th, a third out on a force, batting order across a full game, and the termination guard.
- **Real model:** reproducibility, and legal states checked at every step. A short pilot measures runtime.

In [7]:
def scripted_model(play_rule=forced_outcome, event_rule=NO_EVENT):
    """A deterministic model for rule tests: one fixed outcome per situation, starters never removed, 4 pitches a play."""
    beta = np.zeros(12)
    beta[0] = -50.0
    return {"play": pack({k: [(*play_rule(*v), 1.0)] for k, v in PLAY_KEYS.items()}, len(PLAY_KEYS)),
            "event": pack({k: [(*event_rule(*v), 1.0)] for k, v in EVENT_KEYS.items()}, len(EVENT_KEYS)),
            "beta": beta, "pitches": {"values": np.full(8, 4), "start": np.arange(8), "count": np.ones(8, int)}}


def scripted_inputs(away, home):
    """Each team's nine batters always produce the listed outcome (for example 'K' or 'HR')."""
    p = np.zeros((2, 9, 8))
    for team, outcomes in enumerate([away, home]):
        p[team, np.arange(9), [OUTCOMES.index(o) for o in outcomes]] = 1.0
    return {"p_start": p, "p_pen": p, "usual": np.array([22.0, 22.0])}


def one_step(model, inputs, **fields):
    """Play one step (events, then one plate appearance) from a hand-set state and return the new state."""
    state = new_state(1)
    for name, value in fields.items():
        state[name][:] = value
    return simulate_game(inputs, model, 1, np.random.default_rng(0), check=True, state=state, max_steps=1)["state"]


tests = {}
# Probability checks
league = demo["league"]
probs, _, _ = matchup_probs(demo, demo["bat"].index.get_level_values(0)[:500], ["R"] * 500,
                            demo["pit"].index.get_level_values(0)[:500], ["R"] * 500)
tests["log5 rows are valid probabilities"] = np.allclose(probs.sum(axis=1), 1) and (probs > 0).all()
tests["average batter vs average pitcher returns the league rates"] = np.allclose(log5(league[3], league[3], league[3]), league[3])
tests["a zero count still gets a positive rate"] = (blend(np.array([[0, 5, 0, 3, 1, 0, 0, 11.0]]), league[3], demo["bat_k"]) > 0).all()
tests["latest season weight is 1 for every decay"] = all(max(season_weights(TEST_SEASON, d).values()) == 1 for d in WEIGHT_DECAYS)
tests["2026 profiles use only 2022-2025"] = list(season_weights(TEST_SEASON, 0.8)) == DEV_SEASONS

# Runner-table checks on the 2022-2023 tables used for 2024
model_2024 = make_model(TUNE_SEASON, 1.0, 1.0)
pt = model_2024["play"]
keys = np.array(list(PLAY_KEYS))
masks, outs_at = np.array([v[1] for v in PLAY_KEYS.values()]), np.array([v[2] for v in PLAY_KEYS.values()])
real = pt["cum"][keys] > np.c_[np.zeros(len(keys)), pt["cum"][keys][:, :-1]]   # outcomes with positive probability
starts_empty = np.stack([((masks >> b) & 1) == 0 for b in range(3)], axis=1)[:, None, :]
d = pt["dest"][keys]
tests["runner tables: probabilities sum to 1"] = np.allclose(pt["cum"][:, -1], 1)
tests["runner tables: no runner from an empty base, no two runners on a base"] = bool(
    not (real[:, :, None] & starts_empty & (d[:, :, 1:] != -1)).any()
    and not (real & (np.max([(d == b).sum(axis=2) for b in (1, 2, 3)], axis=0) > 1) & (pt["outs"][keys] + outs_at[:, None] < 3)).any())
tests["runner tables: never more than 3 outs"] = not (real & (pt["outs"][keys] + outs_at[:, None] > 3)).any()
# A batter retired before reaching first can't drive in a run on the third out. Checked on the plays behind the
# tables: strikeouts and outs on balls in play. (After a hit or an error, a batter thrown out advancing can follow a
# run that already scored, which counts, so those plays are excluded from this check.)
retired = plays_all[(plays_all["pre_outs"] == 2) & (plays_all["d0"] == 0) & plays_all["outcome"].isin([K_, IPNH_])
                    & (plays_all["event_type"] != "field_error")]
tests["no run scores when the batter is retired for the third out"] = (retired["runs"] == 0).all()
dp_key = IPNH_ * 24 + 1 * 3 + 0
dp_share = np.diff(np.r_[0, pt["cum"][dp_key]])[pt["outs"][dp_key] == 2].sum()

# Game-rule checks with scripted outcomes
rule = scripted_model()
s = one_step(rule, scripted_inputs(["BB"] * 9, ["K"] * 9), bases=[[6, 7, 8]])
tests["bases-loaded walk forces in one run"] = s["score"][0, 0] == 1 and s["bases"][0].tolist() == [0, 6, 7]
dp_rule = lambda cat, mask, outs: (0, 0, *[b + 1 if (mask >> b) & 1 else -1 for b in (1, 2)], 2, 0) \
    if cat == IPNH_ and mask & 1 and outs < 2 else forced_outcome(cat, mask, outs)
s = one_step(scripted_model(play_rule=dp_rule), scripted_inputs(["IPNH"] * 9, ["K"] * 9), bases=[[4, -1, -1]])
tests["double play: two outs, bases empty"] = s["outs"][0] == 2 and (s["bases"][0] == -1).all()
caught = lambda cat, mask, outs: (-1, 0 if mask & 1 else -1, *[b + 1 if (mask >> b) & 1 else -1 for b in (1, 2)], int(mask & 1), 0)
s = one_step(scripted_model(event_rule=caught), scripted_inputs(["K"] * 9, ["K"] * 9), inning=3, outs=2, bases=[[4, -1, -1]], bat=[[5, 0]])
tests["inning-ending caught stealing: batter doesn't finish and leads off next inning"] = (
    s["half"][0] == 1 and s["outs"][0] == 0 and s["bat"][0, 0] == 5)
s = one_step(rule, scripted_inputs(["K"] * 9, ["K"] * 9), inning=10, bat=[[3, 0]])
tests["extra innings: the previous batter starts on second"] = s["bases"][0, 1] == 2 and s["outs"][0] == 1
home_hits = lambda hit: scripted_inputs(["K"] * 9, [hit] * 9)
s = one_step(rule, home_hits("2B"), inning=9, half=1, outs=1, score=[[3, 3]], bases=[[1, 2, 3]])
tests["walk-off double: only the winning run counts"] = s["over"][0] and s["score"][0].tolist() == [3, 4]
s = one_step(rule, home_hits("HR"), inning=9, half=1, outs=1, score=[[3, 3]], bases=[[1, 2, 3]])
tests["walk-off grand slam: every run counts"] = s["over"][0] and s["score"][0].tolist() == [3, 7]
s = one_step(rule, scripted_inputs(["K"] * 9, ["K"] * 9), inning=9, outs=2, score=[[1, 2]])
tests["home team ahead after the top of the 9th: game over"] = s["over"][0] and s["inning"][0] == 9
s = one_step(rule, scripted_inputs(["K"] * 9, ["K"] * 9), inning=9, outs=2, score=[[2, 1]])
tests["away team ahead after the top of the 9th: bottom half is played"] = not s["over"][0] and s["half"][0] == 1
s = one_step(rule, scripted_inputs(["K"] * 9, ["IPNH"] * 9), half=1, outs=2, bases=[[1, 2, 3]])
tests["third out on a force: no run scores"] = s["score"][0, 1] == 0 and s["half"][0] == 0 and s["inning"][0] == 2
# Whole scripted game: the away leadoff hitter always homers, everyone else strikes out
game = simulate_game(scripted_inputs(["HR"] + ["K"] * 8, ["K"] * 9), rule, 3, np.random.default_rng(1), check=True)
expected_runs, slot = 0, 0
for _ in range(27):                      # 9 innings x 3 outs; each home run is an extra plate appearance
    while slot == 0:
        expected_runs, slot = expected_runs + 1, 1
    slot = (slot + 1) % 9
tests["full game: batting order, scoring and nine innings"] = (game["away"] == expected_runs).all() and (game["innings"] == 9).all() \
    and (game["state"]["bat"][:, 0] == slot).all()
try:
    simulate_game(scripted_inputs(["K"] * 9, ["K"] * 9), rule, 2, np.random.default_rng(2))
    tests["termination guard reports a never-ending tie"] = False
except RuntimeError:
    tests["termination guard reports a never-ending tie"] = True

# Real model: reproducibility and legal states
pilot_games = games.query("season == @TUNE_SEASON")["game_pk"].iloc[:40].tolist()
inputs_2024, _ = game_inputs(pilot_games[:1], model_2024["profile"], model_2024["removal"], model_2024["shares"], False)
one = inputs_2024[pilot_games[0]]
a, b = (simulate_game(one, model_2024, 500, np.random.default_rng([SEED, 1]), check=True) for _ in range(2))
c = simulate_game(one, model_2024, 500, np.random.default_rng([SEED, 2]))
tests["same seed reproduces a game exactly; another seed differs"] = (a["home"] == b["home"]).all() and (a["away"] == b["away"]).all() \
    and not ((a["home"] == c["home"]).all() and (a["away"] == c["away"]).all())
tests["real model: legal states throughout 500 simulations (checked every step)"] = True

display(pd.Series(tests, name="passed").to_frame())
if not all(tests.values()):
    raise AssertionError("failed: " + "; ".join(name for name, ok in tests.items() if not ok))
print(f"in play, no hit with a runner on first and no outs: double play in {dp_share:.1%} of 2022-2023 plays")
print("runner-table cells (own data / borrowed across outs / rule):", model_2024["sources"])

started = time.perf_counter()
pilot, _ = simulate_games(pilot_games, model_2024, "baseline")
per_game = (time.perf_counter() - started) / len(pilot_games)
print(f"pilot: {len(pilot_games)} games x {N_SIMS} simulations, {per_game:.2f} s per game "
      f"(about {per_game * 2430 / 60:.0f} min per season); mean runs per game {(pilot['home_runs_mean'] + pilot['away_runs_mean']).mean():.2f}")

,passed
log5 rows are valid probabilities,True
average batter vs average pitcher returns the league rates,True
a zero count still gets a positive rate,True
latest season weight is 1 for every decay,True
2026 profiles use only 2022-2025,True
runner tables: probabilities sum to 1,True
"runner tables: no runner from an empty base, no two runners on a base",True
runner tables: never more than 3 outs,True
no run scores when the batter is retired for the third out,True
bases-loaded walk forces in one run,True


in play, no hit with a runner on first and no outs: double play in 22.9% of 2022-2023 plays
runner-table cells (own data / borrowed across outs / rule): ({'own': 174, 'pooled': 18, 'rule': 0}, {'own': 21, 'pooled': 0, 'rule': 0})


pilot: 40 games x 1000 simulations, 0.05 s per game (about 2 min per season); mean runs per game 9.12


## Development: 2024 tuning and one 2025 check

1. **Plate-appearance settings.** Three season-weight decays and three blending scales are scored by log loss on 2024 plate appearances, using 2022-2023 only. The lowest wins.
2. **One check on 2025** with those settings and 2022-2024 history. Nothing is changed after it.
3. **Whole games for 2024 and 2025**, baseline and sensitivity bullpens, against two baselines: the home-team win rate from earlier seasons, and last season's records combined by log5 with a home edge. Intervals resample whole game dates, because games on the same day aren't independent.
4. **Recorded plays replayed.** Each recorded half-inning's outcomes are replayed through the runner tables to compare runs per half-inning. A replayed half stops when its recorded plays run out or the replay reaches 3 outs. When the two differ, plays are lost, so the replayed mean is biased low. The share of halves affected is shown.

The baseline bullpen is the main result regardless of how the sensitivity run compares; that was fixed before any results.

In [8]:
def season_results(season):
    """Actual outcomes of completed games in a development season (Final or Completed Early)."""
    r = dev_results.query("season == @season and status != 'Cancelled'").merge(
        games[["game_pk", "official_date", "home_team_id", "away_team_id"]], on="game_pk")
    return r.assign(home_win=(r["home_score"] > r["away_score"]).astype(int), extra=(r["innings_played"] > 9).astype(int))


def baselines(season, schedule):
    """Two simple predictions from earlier seasons: the home-team win rate, and last season's records by log5."""
    earlier = pd.concat([season_results(s) for s in DEV_SEASONS if s < season])
    home_rate = earlier["home_win"].mean()
    last = season_results(season - 1)
    wins = pd.concat([last.assign(team=last["home_team_id"], win=last["home_win"]),
                      last.assign(team=last["away_team_id"], win=1 - last["home_win"])]).groupby("team")["win"].mean()
    h, a = schedule["home_team_id"].map(wins).fillna(0.5), schedule["away_team_id"].map(wins).fillna(0.5)
    matchup = h * (1 - a) / (h * (1 - a) + a * (1 - h))
    return pd.DataFrame({"game_pk": schedule["game_pk"], "p_home_rate": home_rate,
                         "p_last_season": special.expit(special.logit(matchup) + special.logit(home_rate))})


def date_bootstrap(values, dates, seed):
    """Mean and 95% interval of per-game values, resampling whole game dates (games on a date share conditions)."""
    frame = pd.DataFrame(values).groupby(np.asarray(dates)).agg(["sum", "count"])
    sums = frame.xs("sum", axis=1, level=1).to_numpy()
    counts = frame.xs("count", axis=1, level=1).to_numpy()
    pick = np.random.default_rng(seed).integers(0, len(frame), (BOOTSTRAP_DRAWS, len(frame)))
    draws = sums[pick].sum(axis=1) / counts[pick].sum(axis=1)
    point = sums.sum(axis=0) / counts.sum(axis=0)
    low, high = np.percentile(draws, [2.5, 97.5], axis=0)
    return pd.DataFrame({"estimate": point, "ci_low": low, "ci_high": high}, index=list(values))


def score_predictions(p, y, dates, seed):
    """Winner accuracy (a 50% prediction earns half credit), log loss and Brier score, with date-clustered intervals."""
    p, y = np.asarray(p, float), np.asarray(y, float)
    values = {"winner_accuracy": np.where(p == 0.5, 0.5, ((p > 0.5) == y).astype(float)),
              "log_loss": -(y * np.log(p) + (1 - y) * np.log(1 - p)), "brier": (p - y) ** 2}
    return date_bootstrap(values, dates, seed)


def evaluate(sims, season, label, results):
    """Metrics for the simulator and both baselines, plus paired differences (simulator minus baseline)."""
    df = sims.merge(results, on="game_pk").merge(baselines(season, results), on="game_pk")
    rows = []
    for name, col in [("simulator", "p_home"), ("home-team rate", "p_home_rate"), ("last-season records", "p_last_season")]:
        rows.append(score_predictions(df[col], df["home_win"], df["official_date"], SEED).assign(model=name))
    sim_ll = -(df["home_win"] * np.log(df["p_home"]) + (1 - df["home_win"]) * np.log(1 - df["p_home"]))
    for name, col in [("home-team rate", "p_home_rate"), ("last-season records", "p_last_season")]:
        base_ll = -(df["home_win"] * np.log(df[col]) + (1 - df["home_win"]) * np.log(1 - df[col]))
        diff = date_bootstrap({"log_loss_difference": sim_ll - base_ll,
                               "brier_difference": (df["p_home"] - df["home_win"]) ** 2 - (df[col] - df["home_win"]) ** 2},
                              df["official_date"], SEED)
        rows.append(diff.assign(model=f"simulator minus {name}"))
    table = pd.concat(rows).rename_axis("metric").reset_index().assign(season=season, run=label, games=len(df))
    runs = {"simulated runs per game": (df["home_runs_mean"] + df["away_runs_mean"]).mean(),
            "actual runs per game": (df["home_score"] + df["away_score"]).mean(),
            "simulated extra-inning share": df["p_extra_innings"].mean(), "actual extra-inning share": df["extra"].mean(),
            "mean absolute error of run difference": ((df["home_runs_mean"] - df["away_runs_mean"]) - (df["home_score"] - df["away_score"])).abs().mean(),
            "Monte Carlo standard error per game (mean)": np.sqrt(df["p_home"] * (1 - df["p_home"]) / N_SIMS).mean()}
    return table, runs, df


def replay_innings(model, season, seed=SEED):
    """Replay each recorded half-inning's outcomes through the runner tables and compare runs per half-inning."""
    d = pa[pa["season"] == season]
    halves = d.groupby(["game_pk", "inning", "half"], sort=False)
    actual = halves.apply(lambda h: (h["post_bat_score"] - h["start_bat_score"]).sum(), include_groups=False).to_numpy()
    seqs = [h.loc[h["outcome"].notna(), "outcome"].astype(int).to_numpy() for _, h in halves]
    width = max(map(len, seqs))
    seq = np.full((len(seqs), width), -1)
    for r, s_ in enumerate(seqs):
        seq[r, :len(s_)] = s_
    rng = np.random.default_rng(seed)
    outs, runs, used = np.zeros(len(seqs), np.int8), np.zeros(len(seqs), int), np.zeros(len(seqs), int)
    bases = np.full((len(seqs), 3), -1, np.int8)
    bases[halves["inning"].first().to_numpy() >= 10, 1] = 8           # extra-inning runner
    for k in range(width):
        live = np.flatnonzero((outs < 3) & (seq[:, k] >= 0))
        keys = base_mask(bases[live]) * 3 + outs[live]
        e = draw(model["event"], keys, rng.random(len(live)))
        bases[live] = move_runners(bases[live], np.full(len(live), -1, np.int8), model["event"]["dest"][keys, e])
        outs[live] += model["event"]["outs"][keys, e]
        runs[live] += model["event"]["runs"][keys, e]
        live = live[outs[live] < 3]
        used[live] += 1
        keys = seq[live, k] * 24 + base_mask(bases[live]) * 3 + outs[live]
        p = draw(model["play"], keys, rng.random(len(live)))
        bases[live] = move_runners(bases[live], np.full(len(live), k % 8, np.int8), model["play"]["dest"][keys, p])
        outs[live] += model["play"]["outs"][keys, p]
        runs[live] += model["play"]["runs"][keys, p]
    bins = lambda r: pd.Series(np.minimum(r, 5)).value_counts(normalize=True).sort_index().rename({5: "5+"})
    # A replayed half stops when the recorded plays run out or the replay reaches 3 outs; when these differ from the
    # record, some recorded plays go unused or the replay ends short of 3 outs, so the replayed mean runs low.
    lengths = (seq >= 0).sum(axis=1)
    differs = (used < lengths) | (outs < 3)
    table = pd.DataFrame({"replayed": bins(runs), "actual": bins(actual)}).T
    return table.assign(mean=[runs.mean(), actual.mean()], halves_ending_differently=[differs.mean(), np.nan])


# Stage 1: plate-appearance settings, chosen on 2024 using 2022-2023 only
grid = []
for decay in WEIGHT_DECAYS:
    for scale in BLEND_SCALES:
        grid.append({"decay": decay, "blend_scale": scale,
                     "log_loss_2024": pa_log_loss(build_profiles(TUNE_SEASON, decay, scale), TUNE_SEASON)})
grid = pd.DataFrame(grid)
league_only = completed[completed["season"] == TUNE_SEASON]
league_ll = -np.log(demo["league"][split_code(league_only["bat_side"], league_only["pitch_hand"]), league_only["outcome"]]).mean()
best = grid.loc[grid["log_loss_2024"].idxmin()]
DECAY, BLEND_SCALE = float(best["decay"]), float(best["blend_scale"])
# Stage 2: one check on 2025 with the chosen settings (2022-2024 history); nothing is changed after it
check_ll = pa_log_loss(build_profiles(CHECK_SEASON, DECAY, BLEND_SCALE), CHECK_SEASON)
display(grid.pivot(index="decay", columns="blend_scale", values="log_loss_2024").round(5))
print(f"chosen: decay {DECAY}, blending scale {BLEND_SCALE}. Plate-appearance log loss, 2024: {best['log_loss_2024']:.4f} "
      f"(league rates only: {league_ll:.4f}); 2025 check: {check_ll:.4f}")

# Stage 3: whole games. 2024 uses 2022-2023 history, 2025 uses 2022-2024; settings stay fixed.
dev_tables, dev_runs, replays = [], {}, {}
for season in [TUNE_SEASON, CHECK_SEASON]:
    model = make_model(season, DECAY, BLEND_SCALE)
    results = season_results(season)
    replays[season] = replay_innings(model, season)
    started = time.perf_counter()
    for variant in ["baseline", "sensitivity"]:
        sims, _ = simulate_games(results["game_pk"], model, variant)
        table, runs, merged = evaluate(sims, season, variant, results)
        dev_tables.append(table)
        dev_runs[(season, variant)] = runs
    starts_actual = starts_table([season])[1]["bf"].mean()
    dev_runs[(season, "baseline")]["actual starter batters faced"] = starts_actual
    dev_runs[(season, "baseline")]["simulated starter batters faced"] = (
        merged["home_starter_bf_mean"].mean() + merged["away_starter_bf_mean"].mean()) / 2
    print(f"{season}: {len(results)} games simulated twice in {time.perf_counter() - started:.0f} s")

dev_games = pd.concat(dev_tables, ignore_index=True)
display(dev_games.query("run == 'baseline'").pivot_table(index=["model", "metric"], columns="season", values="estimate").round(4))
display(pd.DataFrame(dev_runs).round(3))
for season, table in replays.items():
    print(f"runs per half-inning, {season} plays replayed through tables built from earlier seasons:")
    display(table.round(3))
grid_out = pd.concat([grid.assign(stage="2024 tuning"),
                      pd.DataFrame([{"decay": DECAY, "blend_scale": BLEND_SCALE, "log_loss_2025": check_ll, "stage": "2025 check"},
                                    {"log_loss_2024": league_ll, "stage": "league rates only (reference)"}])], ignore_index=True)
print("saved:", save_output(table_text(grid_out), "development_comparison.csv"), "/",
      save_output(table_text(dev_games.merge(pd.DataFrame(dev_runs).T.rename_axis(["season", "run"]).reset_index(),
                                             on=["season", "run"], how="left")), "development_games.csv"))

blend_scale,0.5,1.0,2.0
decay,,,
0.6,1.47153,1.47036,1.47094
0.8,1.47190,1.47037,1.47062
1.0,1.47240,1.47056,1.47051


chosen: decay 0.6, blending scale 1.0. Plate-appearance log loss, 2024: 1.4704 (league rates only: 1.4861); 2025 check: 1.4694


2024: 2429 games simulated twice in 237 s


2025: 2430 games simulated twice in 240 s


season                                                     2024    2025
model                               metric                             
home-team rate                      brier                0.2496  0.2485
                                    log_loss             0.6923  0.6901
                                    winner_accuracy      0.5216  0.5428
last-season records                 brier                0.2494  0.2458
                                    log_loss             0.6925  0.6847
                                    winner_accuracy      0.5364  0.5519
simulator                           brier                0.2443  0.2465
                                    log_loss             0.6816  0.6860
                                    winner_accuracy      0.5502  0.5395
simulator minus home-team rate      brier_difference    -0.0053 -0.0020
                                    log_loss_difference -0.0107 -0.0041
simulator minus last-season records brier_difference    -0.0051  0.0007
                                    log_loss_difference -0.0109  0.0013

2024                 2025  \
                                           baseline sensitivity baseline   
simulated runs per game                       9.240       9.246    9.032   
actual runs per game                          8.787       8.787    8.895   
simulated extra-inning share                  0.098       0.099    0.101   
actual extra-inning share                     0.089       0.089    0.086   
mean absolute error of run difference         3.445       3.446    3.548   
Monte Carlo standard error per game (mean)    0.016       0.016    0.016   
actual starter batters faced                 22.015         NaN   21.915   
simulated starter batters faced              21.907         NaN   21.992   

                                                        
                                           sensitivity  
simulated runs per game                          9.041  
actual runs per game                             8.895  
simulated extra-inning share                     0.100  
actual extra-inning share                        0.086  
mean absolute error of run difference            3.549  
Monte Carlo standard error per game (mean)       0.016  
actual starter batters faced                       NaN  
simulated starter batters faced                    NaN

runs per half-inning, 2024 plays replayed through tables built from earlier seasons:


,0,1,2,3,4,5+,mean,halves_ending_differently
replayed,0.741,0.146,0.065,0.028,0.013,0.008,0.454,0.224
actual,0.730,0.145,0.069,0.031,0.014,0.010,0.493,NaN


runs per half-inning, 2025 plays replayed through tables built from earlier seasons:


,0,1,2,3,4,5+,mean,halves_ending_differently
replayed,0.739,0.147,0.064,0.030,0.011,0.009,0.46,0.222
actual,0.727,0.148,0.069,0.031,0.014,0.011,0.50,NaN


saved: unchanged / unchanged


## 2026 backtest

The settings are frozen. Every table and the removal model are rebuilt from 2022-2025, and each 2026 game with lineups and starters is simulated 1,000 times with both bullpen versions. The win probability is `(wins + 0.5) / 1,001`, fixed before evaluation so no game gets exactly 0 or 1. The predictions file and its hash are saved here, before any 2026 result is read.

In [9]:
# Settings are frozen here. Profiles, tables, the removal model and bullpen weights are recalculated from all of
# 2022-2025; nothing from 2026 except each game's pregame lineups, starters and bullpen list enters a prediction.
final = make_model(TEST_SEASON, DECAY, BLEND_SCALE)
schedule = games.query("season == @TEST_SEASON")
playable = schedule[schedule["game_pk"].isin(starters["game_pk"]) & schedule["game_pk"].isin(lineups["game_pk"])]
started = time.perf_counter()
base, totals = simulate_games(playable["game_pk"], final, "baseline")
sens, _ = simulate_games(playable["game_pk"], final, "sensitivity")
print(f"simulated {len(base)} of {len(schedule)} scheduled 2026 games x {N_SIMS} twice in {time.perf_counter() - started:.0f} s; "
      f"not simulated (no lineups or starters): {sorted(int(g) for g in set(schedule['game_pk']) - set(playable['game_pk']))}")

predictions = (playable[["game_pk", "official_date", "game_number", "away_team_id", "home_team_id"]]
               .merge(base, on="game_pk")
               .merge(sens[["game_pk", "p_home", "home_runs_mean", "away_runs_mean", "p_extra_innings"]].rename(
                   columns=lambda c: c if c == "game_pk" else f"{c}_sensitivity"), on="game_pk"))
predictions["mc_standard_error"] = np.sqrt(predictions["p_home"] * (1 - predictions["p_home"]) / N_SIMS)
prediction_text = table_text(predictions)
print("predictions:", save_output(prediction_text, "predictions_2026.csv"))
print("simulated total runs per game (all simulations):",
      dict(zip(["mean", "share 0-5", "share 6-10", "share 11+"],
               np.round([np.arange(40) @ totals / totals.sum(), totals[:6].sum() / totals.sum(),
                         totals[6:11].sum() / totals.sum(), totals[11:].sum() / totals.sum()], 3).tolist())))

settings = {
    "seed": SEED, "simulations_per_game": N_SIMS, "smoothing": f"(wins + {SMOOTHING}) / (simulations + {2 * SMOOTHING})",
    "development_seasons": DEV_SEASONS, "tuning_season": TUNE_SEASON, "check_season": CHECK_SEASON, "test_season": TEST_SEASON,
    "event_first_season": EVENT_FIRST_SEASON, "weight_decay": DECAY, "blend_scale": BLEND_SCALE,
    "season_weights": season_weights(TEST_SEASON, DECAY),
    "blend_k_batters": dict(zip(OUTCOMES, final["profile"]["bat_k"].round(1).tolist())),
    "blend_k_pitchers": dict(zip(OUTCOMES, final["profile"]["pit_k"].round(1).tolist())),
    "removal_coefficients": final["beta"].round(6).tolist(), "runner_table_cells": final["sources"],
    "fixed_choices": {"min_play_cell": MIN_PLAY_CELL, "usual_length_prior_starts": USUAL_LENGTH_PRIOR,
                      "relief_share_prior_games": RELIEF_SHARE_PRIOR, "max_innings_guard": MAX_INNINGS,
                      "bootstrap_draws": BOOTSTRAP_DRAWS},
    "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "simulated_total_runs_counts": totals.tolist(),   # all baseline simulations; index = total runs (39 = 39+)
    "predictions_sha256": hashlib.sha256(prediction_text.encode()).hexdigest(),
}
print("settings:", save_output(json.dumps(settings, indent=2) + "\n", "run_settings.json"))

simulated 2429 of 2430 scheduled 2026 games x 1000 twice in 242 s; not simulated (no lineups or starters): [823490]
predictions: unchanged
simulated total runs per game (all simulations): {'mean': 9.054, 'share 0-5': 0.223, 'share 6-10': 0.439, 'share 11+': 0.339}
settings: unchanged


## 2026 evaluation

- **What is measured:**
  - Winner accuracy: the team given more than 50%. An exact 50% would earn half credit.
  - Log loss and Brier score.
  - Calibration, with the number of games in each band.
  - Simulated versus actual runs, run differences and extra innings.
- **Comparison:** everything is compared with the same two baselines, with intervals from resampling dates. Paired differences are simulator minus baseline, so a negative value favors the simulator.
- **Monte Carlo error:** reported per game as a standard error, which is not a confidence interval.
- **Game coverage:**
  - The cancelled game is listed but not scored.
  - Shortened games are scored against the official result, and the metrics are also shown without them.
  - Resumed and flagged games are scored and counted.

The simulator has no home-field advantage beyond batting last. Compare its mean predicted home win probability with the actual home win rate, and the calibration chart, before reading accuracy differences.

In [10]:
# Read the saved predictions back and confirm they are the file recorded before any 2026 result was opened
saved_text = (OUT_DIR / "predictions_2026.csv").read_text()
run_settings = json.loads((OUT_DIR / "run_settings.json").read_text())
if hashlib.sha256(saved_text.encode()).hexdigest() != run_settings["predictions_sha256"]:
    raise RuntimeError("predictions_2026.csv changed after it was saved; rerun the 2026 backtest cell")
pred = pd.read_csv(io.StringIO(saved_text)).drop(columns=["official_date", "game_number", "away_team_id", "home_team_id"])

# 2026 outcomes, used only from here on
results_2026 = read_game_csv(GAME_DIR / "game_results.csv").query("season == @TEST_SEASON").merge(
    games[["game_pk", "official_date", "home_team_id", "away_team_id", "was_resumed"]], on="game_pk")
results_2026 = results_2026.assign(home_win=(results_2026["home_score"] > results_2026["away_score"]).astype(int),
                                   extra=(results_2026["innings_played"] > 9).astype(int))
scored = results_2026[results_2026["status"] != "Cancelled"]
cov = coverage.query("season == @TEST_SEASON").set_index("game_pk")

tables, run_rows = [], {}
sensitivity = pred.drop(columns=["p_home", "home_runs_mean", "away_runs_mean", "p_extra_innings"]).rename(columns=lambda c: c.replace("_sensitivity", ""))
for label, sims, res in [("baseline", pred, scored), ("sensitivity: uncertain relievers included", sensitivity, scored),
                         ("baseline, shortened games excluded", pred, scored[scored["status"] != "Completed Early"])]:
    table, runs, merged = evaluate(sims, TEST_SEASON, label, res)
    tables.append(table)
    run_rows[label] = runs
    if label == "baseline":
        evaluated = merged
evaluation = pd.concat(tables, ignore_index=True)
display(evaluation.query("run == 'baseline'").set_index(["model", "metric"])[["estimate", "ci_low", "ci_high"]].round(4))
display(evaluation.query("run != 'baseline' and model == 'simulator'").set_index(["run", "metric"])[["estimate", "ci_low", "ci_high"]].round(4))
display(pd.DataFrame(run_rows).round(3))

# Monte Carlo error is per game and separate from the season-level intervals above
se = evaluated["mc_standard_error"]
print(f"Monte Carlo standard error of a game's win probability: mean {se.mean():.4f}, max {se.max():.4f} "
      f"(95% interval half-width about {1.96 * se.mean():.3f}); expected inflation of the Brier score from simulation "
      f"noise: {(evaluated['p_home'] * (1 - evaluated['p_home']) / N_SIMS).mean():.5f}")

# Calibration: games grouped by predicted home win probability, with the number of games in each group
edges = np.arange(0.0, 1.0001, 0.05)
evaluated["bin"] = pd.cut(evaluated["p_home"], edges, include_lowest=True)
calibration = evaluated.groupby("bin", observed=True).agg(games=("home_win", "size"), mean_predicted=("p_home", "mean"),
                                                          observed_home_win_rate=("home_win", "mean")).reset_index()
calibration["bin"] = calibration["bin"].astype(str)
display(calibration.round(3))

# Coverage and fallbacks for every scheduled 2026 game
reasons = cov["reasons"].fillna("")
coverage_2026 = pd.DataFrame({"count": {
    "scheduled games": len(schedule), "simulated": len(pred), "scored against a result": len(evaluated),
    "cancelled, not scored": int((results_2026["status"] == "Cancelled").sum()),
    "shortened (Completed Early), scored": int((scored["status"] == "Completed Early").sum()),
    "resumed after suspension, scored": int(scored["was_resumed"].sum()),
    "flagged by collection checks, scored": int(evaluated["game_pk"].map(cov["category"]).eq("flagged").sum()),
    "games with a lineup slot lacking 2022-2025 history (league rates used)": int((pred["batters_without_history"] > 0).sum()),
    "lineup slots with no 2022-2025 history in that split (league rates used)": int(pred["batters_without_history"].sum()),
    "starting pitchers with no 2022-2025 pitching (league rates used)": int(pred["starters_without_history"].sum()),
    "starting pitchers without earlier starts (league usual length used)": int(pred["starters_without_starts"].sum()),
    "bullpens with no eligible reliever (league fallback)": int(pred["empty_bullpens"].sum()),
    "games with uncertain bullpen rows": int((pred["uncertain_listed"] > 0).sum())}})
display(coverage_2026)
flagged = cov[cov["category"] == "flagged"]
print("flagged 2026 games:", {int(g): r for g, r in flagged["reasons"].items()})


def figure_bytes(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=120, metadata={"Software": None})
    plt.close(fig)
    return buffer.getvalue()


fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0.3, 0.75], [0.3, 0.75], color="grey", lw=1, label="perfect calibration")
ax.scatter(calibration["mean_predicted"], calibration["observed_home_win_rate"], s=calibration["games"] / 4, label="simulator (area = games)")
ax.set(xlabel="predicted home win probability", ylabel="observed home win rate", title="2026 calibration")
ax.legend(loc="upper left")
fig_cal = figure_bytes(fig)

counts = np.array(run_settings["simulated_total_runs_counts"])
actual_totals = np.bincount(np.minimum((evaluated["home_score"] + evaluated["away_score"]).astype(int), 39), minlength=40)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(np.arange(30) - 0.2, counts[:30] / counts.sum(), width=0.4, label="simulated")
ax.bar(np.arange(30) + 0.2, actual_totals[:30] / actual_totals.sum(), width=0.4, label="actual 2026")
ax.set(xlabel="total runs in the game", ylabel="share of games", title="Total runs per game, 2026")
ax.legend()
fig_runs = figure_bytes(fig)

statuses = {name: save_output(content, name) for name, content in [
    ("evaluation_2026.csv", table_text(evaluation.merge(pd.DataFrame(run_rows).T.rename_axis("run").reset_index(), on="run", how="left"))),
    ("calibration_2026.csv", table_text(calibration)), ("coverage_2026.csv", table_text(coverage_2026.rename_axis("item").reset_index())),
    ("calibration_2026.png", fig_cal), ("runs_2026.png", fig_runs)]}
print("saved:", statuses)

estimate  ci_low  \
model                               metric                                  
simulator                           winner_accuracy        0.5426  0.5242   
                                    log_loss               0.6887  0.6841   
                                    brier                  0.2478  0.2455   
home-team rate                      winner_accuracy        0.5286  0.5107   
                                    log_loss               0.6915  0.6891   
                                    brier                  0.2492  0.2480   
last-season records                 winner_accuracy        0.5463  0.5267   
                                    log_loss               0.6926  0.6843   
                                    brier                  0.2495  0.2455   
simulator minus home-team rate      log_loss_difference   -0.0028 -0.0081   
                                    brier_difference      -0.0014 -0.0040   
simulator minus last-season records log_loss_difference   -0.0040 -0.0109   
                                    brier_difference      -0.0018 -0.0050   

                                                         ci_high  
model                               metric                        
simulator                           winner_accuracy       0.5604  
                                    log_loss              0.6930  
                                    brier                 0.2499  
home-team rate                      winner_accuracy       0.5487  
                                    log_loss              0.6936  
                                    brier                 0.2502  
last-season records                 winner_accuracy       0.5661  
                                    log_loss              0.7012  
                                    brier                 0.2536  
simulator minus home-team rate      log_loss_difference   0.0020  
                                    brier_difference      0.0010  
simulator minus last-season records log_loss_difference   0.0029  
                                    brier_difference      0.0015

estimate  ci_low  \
run                                       metric                              
sensitivity: uncertain relievers included winner_accuracy    0.5393  0.5208   
                                          log_loss           0.6891  0.6847   
                                          brier              0.2480  0.2458   
baseline, shortened games excluded        winner_accuracy    0.5422  0.5239   
                                          log_loss           0.6888  0.6843   
                                          brier              0.2478  0.2456   

                                                           ci_high  
run                                       metric                    
sensitivity: uncertain relievers included winner_accuracy   0.5577  
                                          log_loss          0.6933  
                                          brier             0.2501  
baseline, shortened games excluded        winner_accuracy   0.5600  
                                          log_loss          0.6931  
                                          brier             0.2500

,baseline,sensitivity: uncertain relievers included,"baseline, shortened games excluded"
simulated runs per game,9.054,9.062,9.054
actual runs per game,8.962,8.962,8.959
simulated extra-inning share,0.100,0.100,0.100
actual extra-inning share,0.086,0.086,0.086
mean absolute error of run difference,3.537,3.539,3.535
Monte Carlo standard error per game (mean),0.016,0.016,0.016


Monte Carlo standard error of a game's win probability: mean 0.0157, max 0.0158 (95% interval half-width about 0.031); expected inflation of the Brier score from simulation noise: 0.00025


,bin,games,mean_predicted,observed_home_win_rate
0,"(0.25, 0.3]",3,0.288,0.000
1,"(0.3, 0.35]",12,0.332,0.333
2,"(0.35, 0.4]",64,0.381,0.438
3,"(0.4, 0.45]",327,0.431,0.483
4,"(0.45, 0.5]",827,0.476,0.496
5,"(0.5, 0.55]",787,0.524,0.557
6,"(0.55, 0.6]",319,0.571,0.592
7,"(0.6, 0.65]",79,0.620,0.608
8,"(0.65, 0.7]",11,0.669,0.818


,count
scheduled games,2430
simulated,2429
scored against a result,2429
"cancelled, not scored",1
"shortened (Completed Early), scored",2
"resumed after suspension, scored",1
"flagged by collection checks, scored",1
games with a lineup slot lacking 2022-2025 history (league rates used),1836
lineup slots with no 2022-2025 history in that split (league rates used),4179
starting pitchers with no 2022-2025 pitching (league rates used),372


flagged 2026 games: {823317: 'starting pitcher'}


saved: {'evaluation_2026.csv': 'unchanged', 'calibration_2026.csv': 'unchanged', 'coverage_2026.csv': 'saved', 'calibration_2026.png': 'unchanged', 'runs_2026.png': 'unchanged'}
